# Stage 2: POS and NER Tagging

### Project Structure

```bash
CS5246Project/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                               
    ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb          
-----------------------------------------------------------------
│   ├─ Stage_2_POS_and_NER_Tagging.ipynb                        │
-----------------------------------------------------------------            
    ├─ Stage_3_Singlish_Normalisation.ipynb
    ├─ Stage_4_Singlish_to_English_Conversion.ipynb     
    ├─ Stage_5_Common_Normalisation.ipynb
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

## To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect import-ipynb contractions

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.3 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=02d73e135e3541d3ef91d45787caef529ba5d01ec3a21d70a8bfabbd367e8009
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


### Change Directory

In [ ]:
import os

# Path
# ----

FOLDER_PATH: str = "/content/drive/MyDrive/CS5246Project/"

os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import pandas as pd
from typing import Any

## NER Tagging Class

In [ ]:
class NERTagger:
    """
    NER tagging class using spaCy for named entity recognition.
    The spaCy model is loaded lazily upon instantiation to optimize resource usage.

    Attributes:
        nlp (spacy.Language): The loaded spaCy language model.
    """

    def __init__(self, model: str = "en_core_web_sm") -> None:
        """
        Initializes the NERTagger with a specified spaCy model.

        Args:
            model (str): The name of the spaCy model to load (e.g., 'en_core_web_sm').
                         Defaults to 'en_core_web_sm'.

        Raises:
            ImportError: If spaCy is not installed.
            OSError: If the specified spaCy model is not found or not downloaded.
        """
        try:
            import spacy
        except ImportError as exc:
            raise ImportError("spaCy is required for NER. Install with: pip install spacy") from exc
        try:
            self.nlp = spacy.load(model)
        except OSError as exc:
            raise OSError(
                f"spaCy model '{model}' not found. Install with: python -m spacy download {model}"
            ) from exc

    def extract_entities(self, text: str) -> list[dict[str, Any]]:
        """
        Extracts named entities from a given text using the loaded spaCy model.

        Args:
            text (str): The input text from which to extract entities.

        Returns:
            list[dict[str, Any]]: A list of dictionaries, where each dictionary represents
                                  a named entity and contains its 'text', 'label',
                                  'start' character offset, and 'end' character offset.
        """
        doc = self.nlp(str(text))
        return [
            {"text": ent.text, "label": ent.label_, "start": ent.start_char, "end": ent.end_char}
            for ent in doc.ents
        ]

    def tag_dataframe(self, df: pd.DataFrame, text_col: str, output_col: str = "entities") -> pd.DataFrame:
        """
        Tags named entities in a specified text column of a pandas DataFrame.

        Args:
            df (pd.DataFrame): The input pandas DataFrame.
            text_col (str): The name of the column in `df` containing the text to be processed.
            output_col (str): The name of the new column to be created in the output DataFrame,
                              which will store the extracted entities. Defaults to 'entities'.

        Returns:
            pd.DataFrame: A new DataFrame with an additional column containing the extracted
                          named entities for each row.
        """
        out = df.copy()
        out[output_col] = out[text_col].astype(str).apply(self.extract_entities)
        return out

### Initialize

In [ ]:
ner_tagger = NERTagger()

## POS TAGGER

In [ ]:
class POSTagger:
    """
    POS tagging class using spaCy for Part-of-Speech tagging.
    The spaCy model is loaded lazily upon instantiation to optimize resource usage.

    Attributes:
        nlp (spacy.Language): The loaded spaCy language model.
    """

    def __init__(self, model: str = "en_core_web_sm") -> None:
        """
        Initializes the POSTagger with a specified spaCy model.

        Args:
            model (str): The name of the spaCy model to load (e.g., 'en_core_web_sm').
                         Defaults to 'en_core_web_sm'.

        Raises:
            ImportError: If spaCy is not installed.
            OSError: If the specified spaCy model is not found or not downloaded.
        """
        try:
            import spacy
        except ImportError as exc:
            raise ImportError("spaCy is required for POS tagging. Install with: pip install spacy") from exc
        try:
            self.nlp = spacy.load(model)
        except OSError as exc:
            raise OSError(
                f"spaCy model '{model}' not found. Install with: python -m spacy download {model}"
            ) from exc

    def tag_text(self, text: str) -> list[dict[str, Any]]:
        """
        Performs Part-of-Speech (POS) tagging on a given text using the loaded spaCy model.

        Args:
            text (str): The input text to be tagged.

        Returns:
            list[dict[str, Any]]: A list of dictionaries, where each dictionary represents
                                  a token and contains its 'token', 'lemma', 'pos' (universal
                                  POS tag), and 'tag' (detailed POS tag).
        """
        doc = self.nlp(str(text))
        return [
            {"token": tok.text, "lemma": tok.lemma_, "pos": tok.pos_, "tag": tok.tag_}
            for tok in doc
            if not tok.is_space
        ]

    def tag_dataframe(self, df: pd.DataFrame, text_col: str, output_col: str = "pos_tags") -> pd.DataFrame:
        """
        Performs POS tagging on a specified text column of a pandas DataFrame.

        Args:
            df (pd.DataFrame): The input pandas DataFrame.
            text_col (str): The name of the column in `df` containing the text to be processed.
            output_col (str): The name of the new column to be created in the output DataFrame,
                              which will store the POS tags. Defaults to 'pos_tags'.

        Returns:
            pd.DataFrame: A new DataFrame with an additional column containing the extracted
                          POS tags for each row.
        """
        out = df.copy()
        out[output_col] = out[text_col].astype(str).apply(self.tag_text)
        return out

### Initialize

In [ ]:
pos_tagger = POSTagger()